# DBSCAN Sanity Check: Does Cluster Count Ever Decrease as a Video Grows?

**Question**: if we take the same video and look at longer and longer truncations of it, does the DBSCAN cluster count (our "dynamicity" proxy) ever *decrease*?

**Method (updated)**: no re-embedding or re-clustering here. `dbscan_frame_cluster_assignments.csv` already has one row per video and one column per frame (`frame 0` ... `frame 75`), holding the cluster label that frame was assigned in the full-video DBSCAN run. For each of two example videos (one with `n_clusters == 4`, one with `n_clusters == 8` in `dbscan_results.csv`), we grow a trailing window frame-by-frame — last 1 frame, last 2 frames, ... up to all 76 — and count how many *distinct* cluster labels appear among the frames currently in the window.

**⚠️ Important caveat — read before trusting this plot**: because the cluster *labels* come from a single, already-computed full-video clustering, and a growing trailing window only ever *adds* frames (never removes one that was already in), the set of distinct labels represented in the window can mathematically only grow or stay the same as the window grows — it can never shrink. In other words, **this version of the check is monotonic by construction, regardless of whether DBSCAN's own clustering behavior is well-behaved under truncation.** It verifies something real (that the full-video clustering is internally consistent / that we're slicing it correctly) but it does *not* test whether DBSCAN, if actually *re-run* on a truncated sub-clip in isolation, would still find a non-decreasing cluster count — that would require rerunning DBSCAN independently per truncation length (the embedding-based version of this notebook, kept below in git history / the previous version, does that). If the point of the sanity check is specifically to catch DBSCAN behaving inconsistently when it sees less context, this simplified version can't catch that by design — flagging this so you can decide which question you actually need answered.

**Assumptions made here — double check against the real files**:
- `dbscan_frame_cluster_assignments.csv`: a `video` column (bare video ID, no `.mp4`) plus one column per frame, chronological order (`frame 0` = start of the clip, `frame 75` = end).
- Noise convention: label `-1` = DBSCAN noise (standard sklearn convention, consistent with `dbscan_results.csv` tracking `n_noise_points` separately from `n_clusters`); missing/`NaN` = video shorter than 76 frames, no real frame at that position.
- Frames were sampled at 5Hz (one every 200ms), so frame index ÷ 5 = seconds — used only to label the x-axis in seconds; adjust if the real sampling rate differs.
- This still needs `/braintree/...` access, but no GPU / no video decoding anymore — much cheaper, can run on a login node.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Config

In [ ]:
DBSCAN_RESULTS_CSV = "/braintree/data2/active/users/aicha/results/entire_dataset/dbscan_analysis_outputs/dbscan_results.csv"
FRAME_ASSIGNMENTS_CSV = "/braintree/data2/active/users/aicha/results/entire_dataset/dbscan_analysis_outputs/dbscan_frame_cluster_assignments.csv"

SAMPLING_HZ = 5  # one frame every 200ms -- only used to label the x-axis in seconds
NOISE_LABEL = -1

TARGET_N_CLUSTERS = [4, 8]  # pick one video with each of these cluster counts

## Load both CSVs and pick two example videos

Inspecting `.columns`/`.head()` first for both files — if the real column names/format differ from what's assumed above, this will make it obvious immediately.

In [ ]:
dbscan_df = pd.read_csv(DBSCAN_RESULTS_CSV)
print(dbscan_df.columns.tolist())
dbscan_df.head()

In [ ]:
frames_df = pd.read_csv(FRAME_ASSIGNMENTS_CSV)
print(frames_df.columns.tolist())
frames_df.head()

In [ ]:
# detect the frame columns robustly (handles "frame 0", "frame_0", "Frame0", ...)
# and sort them into chronological order (frame 0 first, last frame last).
frame_col_pattern = re.compile(r"frame[\s_]*?(\d+)", re.IGNORECASE)
frame_cols = [c for c in frames_df.columns if frame_col_pattern.search(c)]
frame_cols = sorted(frame_cols, key=lambda c: int(frame_col_pattern.search(c).group(1)))

print(f"Found {len(frame_cols)} frame columns: {frame_cols[0]} ... {frame_cols[-1]}")

In [ ]:
example_videos = {}
for target in TARGET_N_CLUSTERS:
    matches = dbscan_df.loc[dbscan_df["n_clusters"] == target]
    assert len(matches) > 0, f"No video found with n_clusters == {target}"
    video_id = matches.iloc[0]["video"]

    assert (frames_df["video"] == video_id).any(), (
        f"video={video_id} (n_clusters={target} in {DBSCAN_RESULTS_CSV}) "
        f"not found in {FRAME_ASSIGNMENTS_CSV} -- check the 'video' ID format matches between the two files."
    )

    example_videos[target] = video_id
    print(f"n_clusters={target}  ->  video={video_id}")

example_videos

## Run the sanity check: grow each video from the end, frame by frame

In [ ]:
def cluster_and_noise_counts(labels):
    """labels: 1D array of per-frame cluster labels (may contain NaN for missing frames)."""
    real = labels[~pd.isna(labels)]
    n_clusters = len(set(real) - {NOISE_LABEL})
    n_noise = int((real == NOISE_LABEL).sum())
    return n_clusters, n_noise


results = []

for target_n_clusters, video_id in example_videos.items():
    row = frames_df.loc[frames_df["video"] == video_id, frame_cols].iloc[0]
    labels_chronological = row.to_numpy(dtype=float)  # frame 0 (start) ... frame N-1 (end)

    # how many of the frame columns are actually real (non-NaN) for this video
    n_real_frames = int((~pd.isna(labels_chronological)).sum())

    for n_frames_kept in range(1, n_real_frames + 1):
        # keep the LAST n_frames_kept real frames -- truncation removes from
        # the beginning, real content stays anchored at the end (matches the
        # model's own padding/segment convention)
        window = labels_chronological[n_real_frames - n_frames_kept : n_real_frames]

        n_clusters, n_noise = cluster_and_noise_counts(window)
        results.append({
            "video": video_id,
            "original_n_clusters": target_n_clusters,
            "n_frames_kept": n_frames_kept,
            "truncation_sec": n_frames_kept / SAMPLING_HZ,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
        })

results_df = pd.DataFrame(results).sort_values(["video", "n_frames_kept"]).reset_index(drop=True)
results_df.to_csv("dbscan_sanity_check_results.csv", index=False)
results_df

## Plot: cluster count vs. truncation duration, one panel per video

In [ ]:
fig, axes = plt.subplots(1, len(example_videos), figsize=(7 * len(example_videos), 5), squeeze=False)
axes = axes[0]

for ax, (target_n_clusters, video_id) in zip(axes, example_videos.items()):
    sub = results_df.loc[results_df["video"] == video_id]
    x = np.arange(len(sub))
    width = 0.35

    bars_clusters = ax.bar(x - width / 2, sub["n_clusters"], width, label="Number of clusters")
    bars_noise = ax.bar(x + width / 2, sub["n_noise"], width, label="Number of noise points")

    for bars in (bars_clusters, bars_noise):
        ax.bar_label(bars, padding=2)

    # thin the x-tick labels if there are many frames, to keep them legible
    tick_stride = max(1, len(sub) // 15)
    ax.set_xticks(x[::tick_stride])
    ax.set_xticklabels([f"{t:.1f}s" for t in sub["truncation_sec"].iloc[::tick_stride]], rotation=45, ha="right")
    ax.set_xlabel("Truncation duration (kept from the end of the video)")
    ax.set_ylabel("Count")
    ax.set_title(f"video={video_id}  (full-video n_clusters={target_n_clusters})")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("DBSCAN sanity check: cluster count vs. truncation duration")
fig.tight_layout()
fig.savefig("dbscan_sanity_check_plot.png", dpi=150)
plt.show()

## Check monotonicity explicitly

Given the caveat above, expect this to report "MONOTONIC" for every video by construction — it's a check that the slicing/counting logic is correct, not (on its own) a check of DBSCAN's re-clustering behavior.

In [ ]:
for video_id, sub in results_df.groupby("video"):
    sub = sub.sort_values("n_frames_kept")
    diffs = np.diff(sub["n_clusters"].values)
    n_decreases = int((diffs < 0).sum())
    status = "MONOTONIC (non-decreasing)" if n_decreases == 0 else f"NOT monotonic -- {n_decreases} decrease(s)"
    print(f"video={video_id}: {status}")
    if n_decreases > 0:
        drop_points = sub.iloc[1:][diffs < 0]
        print(drop_points[["truncation_sec", "n_clusters"]])